# ICE & Immigration Law Monitor (2026)

Live monitoring notebook tracking Trump-era immigration enforcement — court rulings, enforcement incidents, policy overreach, and due-process violations.

### What this notebook does
1. Defines **enforcement domains** (ICE, CBP, DOJ-NS, EOIR, ORR, Sanctuary jurisdictions)
2. Fetches and filters RSS feeds from legal watchdogs, courts, ACLU, investigative journalists
3. Provides tracker DataFrames for **court rulings**, **enforcement incidents**, and **policy actions**
4. Renders an overreach-frequency bar chart and enforcement entity network graph
5. Exports a self-contained interactive HTML dashboard
6. Writes `weekly_brief_input.txt` for the NarroVue Immigration Weekly Brief Generator

### Overreach Framework
Each tracked item is tagged with one or more **overreach categories**:
- `DUE_PROCESS` — denial of hearing, counsel, or notice
- `4TH_AMEND` — warrantless searches, unlawful seizure
- `1ST_AMEND` — retaliation for speech, press suppression
- `SEPARATION_OF_POWERS` — defiance of court orders, unilateral executive action
- `TREATY_VIOLATION` — Convention Against Torture, non-refoulement
- `STATUTORY` — violation of INA, APA, or 14th Amendment
- `STATE_PREEMPTION` — unlawful federal pressure on state/local governments

> **v1.0** — Initial instrument modeled after NarroVue Public Health Compacts Monitor architecture.

In [ ]:
# Dependencies
!pip install -q feedparser beautifulsoup4 lxml requests urllib3 networkx matplotlib pandas

In [ ]:
import json, datetime, pathlib, email.utils, textwrap, ssl, random, urllib3
import pandas as pd
import feedparser
import networkx as nx
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from bs4 import BeautifulSoup
from lxml import etree
from urllib.parse import urljoin

urllib3.disable_warnings()

OUTPUT_HTML  = pathlib.Path("ice_immigration_dashboard.html")
BRIEF_TXT    = pathlib.Path("weekly_brief_input.txt")
GENERATED_AT = datetime.datetime.now().strftime("%Y-%m-%d %H:%M")

MAX_ENTRIES_PER_FEED = 15
FETCH_TIMEOUT        = 30

IMMIGRATION_KEYWORDS = [
    "immigration", "ICE", "CBP", "deportation", "detention", "asylum",
    "immigrant", "undocumented", "border", "visa", "DACA", "TPS",
    "removal", "sanctuary", "due process", "habeas", "warrant",
    "Fourth Amendment", "First Amendment", "executive order",
    "court order", "injunction", "overreach", "Alien Enemies Act",
    "non-refoulement", "Convention Against Torture", "INA",
    "immigration court", "EOIR", "BIA", "parole", "withholding",
    "expedited removal", "family separation", "Guantanamo",
    "CECOT", "rendition", "APA", "noncitizen", "migrant",
    "refugee", "travel ban", "birthright citizenship",
]

print("imports OK —", GENERATED_AT)

In [ ]:
USER_AGENT = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/124.0 Safari/537.36"

def build_session():
    session = requests.Session()
    retries = Retry(total=3, backoff_factor=1, status_forcelist=[429,500,502,503,504], allowed_methods=["GET"])
    adapter = HTTPAdapter(max_retries=retries)
    session.mount("http://", adapter); session.mount("https://", adapter)
    session.headers.update({"User-Agent": USER_AGENT, "Accept": "application/rss+xml,application/xml,text/xml,text/html", "Accept-Language": "en-US,en;q=0.9"})
    return session

def recover_malformed_xml(raw):
    try:
        parser = etree.XMLParser(recover=True)
        tree   = etree.fromstring(raw, parser=parser)
        return etree.tostring(tree)
    except Exception:
        return raw

def discover_feed_from_html(html, base_url):
    soup = BeautifulSoup(html, "html.parser")
    found = []
    for link in soup.find_all("link"):
        lt = (link.get("type") or "").lower()
        href = link.get("href")
        if href and ("rss" in lt or "atom" in lt or "xml" in lt):
            found.append(urljoin(base_url, href))
    return found

print("fetcher helpers ready")

## 1 · Enforcement Entity Data

In [ ]:
ENTITY_RECORDS = [
    {
        "Entity": "Immigration and Customs Enforcement (ICE)",
        "Abbreviation": "ICE", "Parent": "DHS",
        "Primary Role": "Interior enforcement, detention, removal operations",
        "Overreach Flags": "Warrantless workplace raids, courthouse arrests, expansion of expedited removal, detention without bond",
        "Key Legal Constraints": "INA §236, §241; 4th/5th Amendments; APA; Flores Settlement; TVPA",
        "Active Litigation": "Multiple ACLU/CLINIC habeas petitions; DOJ court-order defiance cases",
        "Website": "https://www.ice.gov/",
    },
    {
        "Entity": "Customs and Border Protection (CBP)",
        "Abbreviation": "CBP", "Parent": "DHS",
        "Primary Role": "Border enforcement, ports of entry, Title 42/8 expulsions",
        "Overreach Flags": "Turnbacks without asylum screening, denial of asylum access, 100-mile border zone warrantless search claims",
        "Key Legal Constraints": "INA §208, §235; Convention Against Torture; non-refoulement; 4th Amendment border doctrine",
        "Active Litigation": "Al Otro Lado v. Mayorkas; MSI turnback cases",
        "Website": "https://www.cbp.gov/",
    },
    {
        "Entity": "Executive Office for Immigration Review (EOIR)",
        "Abbreviation": "EOIR", "Parent": "DOJ",
        "Primary Role": "Immigration courts and Board of Immigration Appeals",
        "Overreach Flags": "AG self-referral to override BIA precedent, case quotas, denial of continuances",
        "Key Legal Constraints": "INA §240; Due Process Clause; APA; independent adjudicator doctrine",
        "Active Litigation": "Challenges to AG self-referral authority; court-stripping provisions",
        "Website": "https://www.justice.gov/eoir",
    },
    {
        "Entity": "Office of Refugee Resettlement (ORR)",
        "Abbreviation": "ORR", "Parent": "HHS",
        "Primary Role": "Unaccompanied minors, refugee placement and reception",
        "Overreach Flags": "Transfer of UCs to DHS custody, suspension of R&P programs, funding freezes for resettlement contractors",
        "Key Legal Constraints": "Flores Settlement; TVPRA; Refugee Act of 1980",
        "Active Litigation": "Flores monitor proceedings; R&P contractor termination suits",
        "Website": "https://www.acf.hhs.gov/orr",
    },
    {
        "Entity": "DOJ National Security / Immigration Task Forces",
        "Abbreviation": "DOJ-NS", "Parent": "DOJ",
        "Primary Role": "Alien Enemies Act invocations, denaturalization, civil rights reversals",
        "Overreach Flags": "Alien Enemies Act (1798) for non-wartime deportation, denaturalization unit, retaliation against sanctuary jurisdictions",
        "Key Legal Constraints": "AEA statutory scope; Due Process; Separation of Powers; 1st Amendment",
        "Active Litigation": "J.G.G. v. Trump (AEA); Mahmoud Khalil case; sanctuary city funding cases",
        "Website": "https://www.justice.gov/",
    },
    {
        "Entity": "State & Local Resistance / Sanctuary Jurisdictions",
        "Abbreviation": "SANCTUARY", "Parent": "N/A",
        "Primary Role": "Limiting cooperation with federal immigration enforcement",
        "Overreach Flags": "Federal threats to withhold Byrne JAG / Title I funding, DOJ subpoenas of sanctuary city officials",
        "Key Legal Constraints": "Anti-commandeering doctrine (Printz); 10th Amendment; spending clause limits",
        "Active Litigation": "Chicago v. Sessions; California v. Trump; NY/NJ/CT sanctuary funding cases",
        "Website": "",
    },
]

entity_df = pd.DataFrame(ENTITY_RECORDS)
display_cols = ["Abbreviation", "Parent", "Primary Role", "Active Litigation"]
print(f"✓ {len(entity_df)} enforcement entities loaded")
entity_df[display_cols]

## 2 · RSS / News Feed Monitoring

In [ ]:
RSS_FEEDS = [
    # Federal / Official
    {"label": "ICE Newsroom",           "url": "https://www.ice.gov/rss.xml",                                           "category": "Federal"},
    {"label": "CBP Newsroom",           "url": "https://www.cbp.gov/newsroom/rss.xml",                                  "category": "Federal"},
    {"label": "DOJ News",               "url": "https://www.justice.gov/feed/",                                         "category": "Federal"},
    {"label": "DHS Newsroom",           "url": "https://www.dhs.gov/rss",                                               "category": "Federal"},
    {"label": "Federal Register – DHS", "url": "https://www.federalregister.gov/agencies/homeland-security.rss",         "category": "Federal"},
    {"label": "Federal Register – DOJ", "url": "https://www.federalregister.gov/agencies/justice-department.rss",        "category": "Federal"},
    # Legal Watchdogs
    {"label": "ACLU News",              "url": "https://www.aclu.org/news/rss.xml",                                     "category": "Legal"},
    {"label": "ACLU Immigrants Rights", "url": "https://www.aclu.org/topic/immigrants-rights/rss.xml",                  "category": "Legal"},
    {"label": "CLINIC",                 "url": "https://cliniclegal.org/feed",                                          "category": "Legal"},
    {"label": "NILC",                   "url": "https://www.nilc.org/feed/",                                            "category": "Legal"},
    {"label": "Human Rights First",     "url": "https://humanrightsfirst.org/feed/",                                    "category": "Legal"},
    {"label": "Human Rights Watch US",  "url": "https://www.hrw.org/rss/united-states",                                 "category": "Legal"},
    {"label": "Amnesty International US","url": "https://www.amnestyusa.org/feed/",                                     "category": "Legal"},
    {"label": "RAICES",                 "url": "https://www.raicestexas.org/feed/",                                     "category": "Legal"},
    {"label": "Brennan Center",         "url": "https://www.brennancenter.org/rss.xml",                                 "category": "Legal"},
    {"label": "Just Security",          "url": "https://www.justsecurity.org/feed/",                                    "category": "Legal"},
    {"label": "Lawfare Blog",           "url": "https://www.lawfaremedia.org/feed",                                     "category": "Legal"},
    # Courts
    {"label": "SCOTUSblog",             "url": "https://www.scotusblog.com/feed/",                                      "category": "Courts"},
    {"label": "SCOTUSblog Immigration", "url": "https://www.scotusblog.com/tag/immigration/feed/",                       "category": "Courts"},
    {"label": "9th Circuit",            "url": "https://www.ca9.uscourts.gov/rss/",                                     "category": "Courts"},
    {"label": "2nd Circuit",            "url": "https://www.ca2.uscourts.gov/rss.xml",                                  "category": "Courts"},
    # Policy Research
    {"label": "Migration Policy Institute","url": "https://www.migrationpolicy.org/rss.xml",                            "category": "Policy"},
    {"label": "Cato Immigration",        "url": "https://www.cato.org/rss/immigration",                                 "category": "Policy"},
    {"label": "Vera Institute",          "url": "https://www.vera.org/feed",                                            "category": "Policy"},
    {"label": "TRAC Immigration",        "url": "https://trac.syr.edu/rss/immigration.rss",                             "category": "Policy"},
    {"label": "CAP Immigration",         "url": "https://www.americanprogress.org/tag/immigration/feed/",                "category": "Policy"},
    # Journalism / Investigative
    {"label": "ProPublica",              "url": "https://www.propublica.org/feeds/propublica/main",                      "category": "Journalism"},
    {"label": "NYT Immigration",         "url": "https://rss.nytimes.com/services/xml/rss/nyt/ImmigrationImmigration.xml","category": "Journalism"},
    {"label": "Guardian Immigration",    "url": "https://www.theguardian.com/us-news/immigration/rss",                   "category": "Journalism"},
    {"label": "AP Immigration",          "url": "https://feeds.apnews.com/apnews/immigration",                          "category": "Journalism"},
    {"label": "The Intercept",           "url": "https://theintercept.com/feed/?rss",                                   "category": "Journalism"},
    {"label": "Texas Tribune Immigration","url": "https://www.texastribune.org/immigration/rss.xml",                    "category": "Journalism"},
    {"label": "LA Times Immigration",    "url": "https://www.latimes.com/topic/immigration/rss2.0.xml",                  "category": "Journalism"},
    {"label": "Mother Jones",            "url": "https://www.motherjones.com/feed/",                                    "category": "Journalism"},
    {"label": "Documented NY",           "url": "https://documentedny.com/feed/",                                       "category": "Journalism"},
    {"label": "Imprint News",            "url": "https://imprintnews.org/feed/",                                        "category": "Journalism"},
    # Detention / Conditions
    {"label": "Detention Watch Network", "url": "https://www.detentionwatchnetwork.org/news/rss",                        "category": "Detention"},
    {"label": "National Immigrant Justice Center","url": "https://immigrantjustice.org/feed",                          "category": "Detention"},
    {"label": "Physicians for Human Rights","url": "https://phr.org/feed/",                                            "category": "Detention"},
    # State / AGs
    {"label": "NY AG",                   "url": "https://ag.ny.gov/press-releases/rss",                                 "category": "State"},
    {"label": "CA AG",                   "url": "https://oag.ca.gov/news/press-releases/rss",                           "category": "State"},
    {"label": "NJ AG",                   "url": "https://www.njoag.gov/news/rss/",                                      "category": "State"},
    {"label": "National Governors Association","url": "https://www.nga.org/rss/",                                       "category": "State"},
    {"label": "NCSL Immigration",        "url": "https://www.ncsl.org/rss?topic=immigration",                           "category": "State"},
]

print(f"✓ {len(RSS_FEEDS)} feed sources configured")
pd.DataFrame(RSS_FEEDS)

In [ ]:
def parse_published(raw):
    if not raw: return None
    try: return email.utils.parsedate_to_datetime(raw).replace(tzinfo=None)
    except: pass
    for fmt in ("%a, %d %b %Y %H:%M:%S %z","%Y-%m-%dT%H:%M:%SZ","%Y-%m-%dT%H:%M:%S%z","%Y-%m-%d %H:%M:%S"):
        try: return datetime.datetime.strptime(raw.strip(), fmt).replace(tzinfo=None)
        except ValueError: pass
    return None

_kw_lower = [k.lower() for k in IMMIGRATION_KEYWORDS]

def is_relevant(title, summary=""):
    haystack = (title + " " + summary).lower()
    return any(kw in haystack for kw in _kw_lower)

def fetch_feed(feed_info):
    url, label = feed_info["url"], feed_info.get("label", feed_info["url"])
    entries, session = [], build_session()
    try:
        resp = session.get(url, timeout=FETCH_TIMEOUT, verify=False, allow_redirects=True)
        ct, raw = resp.headers.get("Content-Type",""), resp.content
        parsed = feedparser.parse(raw)
        if not parsed.entries:
            parsed = feedparser.parse(recover_malformed_xml(raw))
        if not parsed.entries and "html" in ct.lower():
            for f_url in discover_feed_from_html(raw, url)[:2]:
                alt = session.get(f_url, timeout=FETCH_TIMEOUT, verify=False)
                parsed = feedparser.parse(alt.content)
                if parsed.entries: break
        for entry in parsed.entries[:MAX_ENTRIES_PER_FEED]:
            t, s = entry.get("title",""), entry.get("summary","")
            if not is_relevant(t, s): continue
            rd = entry.get("published","")
            entries.append({"source": label, "category": feed_info.get("category",""),
                            "title": t, "link": entry.get("link",""),
                            "summary": s[:400], "raw_date": rd, "published": parse_published(rd)})
    except: pass
    marker = "✓" if entries else "–"
    print(f"  {marker}  {label}: {len(entries) if entries else 0} items{'' if entries else ' / skipped'}")
    return entries

print("Fetching feeds…")
all_entries = []
for feed in RSS_FEEDS:
    all_entries.extend(fetch_feed(feed))

news_df = (pd.DataFrame(all_entries)
           .drop_duplicates(subset="link", keep="first")
           .sort_values("published", ascending=False, na_position="last")
           .reset_index(drop=True))

print(f"\n✓ {len(news_df)} unique relevant articles across {len(RSS_FEEDS)} feeds")

## 3 · Manual Trackers

Fill in these DataFrames with hand-curated entries each week.

In [ ]:
# Court Rulings Tracker — add rows as cases develop
COURT_RECORDS = [
    {
        "Date": "2025-04-07",
        "Case": "J.G.G. v. Trump",
        "Court": "D.D.C. / SCOTUS",
        "Subject": "Alien Enemies Act invocation for Venezuelan TdA deportations",
        "Ruling": "SCOTUS temporarily blocked deportations; required 'reasonable time' notice; AEA applicability still contested",
        "Overreach Tags": "DUE_PROCESS, STATUTORY, SEPARATION_OF_POWERS",
        "Outcome": "Partial block — litigation ongoing",
        "Source": "https://www.scotusblog.com/case-files/cases/trump-v-j-g-g/",
    },
    {
        "Date": "2025-03-27",
        "Case": "Noem v. Hassan (Mahmoud Khalil)",
        "Court": "D.N.H.",
        "Subject": "Detention of Columbia student for pro-Palestinian speech",
        "Ruling": "Habeas jurisdiction confirmed; detention challenged as viewpoint-based retaliation",
        "Overreach Tags": "1ST_AMEND, DUE_PROCESS",
        "Outcome": "Ongoing — venue disputed",
        "Source": "https://www.aclu.org/cases/mahmoud-khalil-v-trump",
    },
    # Add rows below
]
court_df = pd.DataFrame(COURT_RECORDS)
print(f"✓ {len(court_df)} court rulings loaded")
court_df

In [ ]:
# Enforcement Incidents Tracker
INCIDENT_RECORDS = [
    {
        "Date": "2025-01-26",
        "Entity": "ICE",
        "Location": "Nationwide",
        "Incident Type": "Mass Workplace Raid",
        "Description": "Coordinated raids at food processing, agricultural, and construction sites in 10+ states; 500+ arrests; attorneys denied access at sites",
        "Overreach Tags": "4TH_AMEND, DUE_PROCESS",
        "Legal Challenge": "ACLU emergency motions filed in multiple districts",
        "Source": "",
    },
    {
        "Date": "2025-03-15",
        "Entity": "ICE / DOJ",
        "Location": "El Salvador (CECOT)",
        "Incident Type": "Third-Country Removal",
        "Description": "238 Venezuelan nationals transferred to CECOT mega-prison without individualized hearings; classified as Tren de Aragua despite disputed gang evidence",
        "Overreach Tags": "DUE_PROCESS, TREATY_VIOLATION, STATUTORY",
        "Legal Challenge": "J.G.G. v. Trump; multiple habeas petitions",
        "Source": "",
    },
    # Add rows below
]
incident_df = pd.DataFrame(INCIDENT_RECORDS)
print(f"✓ {len(incident_df)} incidents loaded")
incident_df

In [ ]:
# Policy Actions Tracker
POLICY_RECORDS = [
    {
        "Date": "2025-01-20",
        "Entity": "White House / DHS",
        "Policy Area": "Expedited Removal Expansion",
        "Action": "EO directing DHS to apply expedited removal nationwide; removes immigration judge review for anyone in U.S. fewer than 2 years",
        "Overreach Tags": "DUE_PROCESS, STATUTORY, SEPARATION_OF_POWERS",
        "Federal Interaction": "Expands INA §235(b)(1) beyond prior geographic limits; ACLU challenges statutory authority",
        "States Affected": "All",
        "Source": "",
    },
    {
        "Date": "2025-01-20",
        "Entity": "White House",
        "Policy Area": "Birthright Citizenship EO",
        "Action": "EO purporting to end birthright citizenship for children of undocumented parents; immediately enjoined by multiple federal courts",
        "Overreach Tags": "STATUTORY, SEPARATION_OF_POWERS",
        "Federal Interaction": "14th Amendment violation; multiple circuits enjoined; SCOTUS heard arguments May 2025",
        "States Affected": "All",
        "Source": "",
    },
    {
        "Date": "2025-02-10",
        "Entity": "DOJ",
        "Policy Area": "Sanctuary Jurisdiction Defunding",
        "Action": "DOJ memo threatening to withhold Byrne JAG, Title I, and other federal grants from jurisdictions limiting ICE cooperation; subpoenas issued to officials in Chicago, NYC, and LA",
        "Overreach Tags": "STATE_PREEMPTION, SEPARATION_OF_POWERS",
        "Federal Interaction": "Anti-commandeering doctrine (Printz v. United States); spending clause limits (NFIB v. Sebelius)",
        "States Affected": "IL, NY, CA, NJ, CT, MA, WA, OR, CO",
        "Source": "",
    },
    # Add rows below
]
policy_df = pd.DataFrame(POLICY_RECORDS)
print(f"✓ {len(policy_df)} policy actions loaded")
policy_df

## 4 · Overreach Heatmap

In [ ]:
import re
from collections import Counter

OVERREACH_CATEGORIES = ["DUE_PROCESS","4TH_AMEND","1ST_AMEND","SEPARATION_OF_POWERS","TREATY_VIOLATION","STATUTORY","STATE_PREEMPTION"]

def count_tags(df, col="Overreach Tags"):
    counter = Counter()
    if col not in df.columns: return counter
    for val in df[col].dropna():
        for tag in re.split(r'[,;\s]+', val):
            tag = tag.strip()
            if tag in OVERREACH_CATEGORIES: counter[tag] += 1
    return counter

combined = Counter()
for df_t in [court_df, incident_df, policy_df]:
    combined += count_tags(df_t)

cats   = OVERREACH_CATEGORIES
counts = [combined.get(c, 0) for c in cats]
colors = ["#c0392b" if v > 0 else "#dcdcdc" for v in counts]

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(cats, counts, color=colors, width=0.6)
ax.set_title("Overreach Category Frequency", fontsize=13, fontweight="bold", pad=12)
ax.set_ylabel("Occurrences")
# Fix: use set_xticks + set_xticklabels together to avoid FixedLocator warning
ax.set_xticks(range(len(cats)))
ax.set_xticklabels(cats, rotation=30, ha="right")
ax.spines[["top","right"]].set_visible(False)
for bar, v in zip(bars, counts):
    if v > 0: ax.text(bar.get_x()+bar.get_width()/2, v+0.05, str(v), ha="center", va="bottom", fontsize=10)
plt.tight_layout()
plt.savefig("overreach_heatmap.png", dpi=120, bbox_inches="tight")
plt.close()  # close instead of show() — avoids non-interactive canvas warning
print("overreach heatmap saved")


## 5 · Entity Network Graph

In [ ]:
G = nx.DiGraph()
for abbr in entity_df["Abbreviation"].tolist(): G.add_node(abbr, node_type="entity")
for cat in OVERREACH_CATEGORIES: G.add_node(cat, node_type="overreach")

EDGES = [("ICE","DUE_PROCESS"),("ICE","4TH_AMEND"),("CBP","TREATY_VIOLATION"),("CBP","DUE_PROCESS"),
         ("EOIR","DUE_PROCESS"),("EOIR","SEPARATION_OF_POWERS"),("ORR","STATUTORY"),("ORR","DUE_PROCESS"),
         ("DOJ-NS","STATUTORY"),("DOJ-NS","1ST_AMEND"),("DOJ-NS","SEPARATION_OF_POWERS"),
         ("SANCTUARY","STATE_PREEMPTION"),("SANCTUARY","1ST_AMEND")]
G.add_edges_from(EDGES)

fig, ax = plt.subplots(figsize=(13, 8))
pos = nx.spring_layout(G, seed=42, k=2.2)
entity_nodes    = [n for n,d in G.nodes(data=True) if d.get("node_type")=="entity"]
overreach_nodes = [n for n,d in G.nodes(data=True) if d.get("node_type")=="overreach"]
nx.draw_networkx_nodes(G, pos, nodelist=entity_nodes, node_color="#1a3a5c", node_size=900, ax=ax)
nx.draw_networkx_nodes(G, pos, nodelist=overreach_nodes, node_color="#c0392b", node_size=600, ax=ax)
nx.draw_networkx_edges(G, pos, edge_color="#888", arrows=True, arrowstyle="->", arrowsize=18, ax=ax)
nx.draw_networkx_labels(G, pos, font_size=8, font_color="white", ax=ax)
ax.legend(handles=[mpatches.Patch(facecolor="#1a3a5c",label="Enforcement Entity"),
                   mpatches.Patch(facecolor="#c0392b",label="Overreach Category")], loc="upper left", fontsize=9)
ax.set_title("Enforcement Entity -> Overreach Category Network", fontsize=13, fontweight="bold")
ax.axis("off")
plt.tight_layout()
plt.savefig("entity_network.png", dpi=120, bbox_inches="tight")
plt.close()  # close instead of show() — avoids non-interactive canvas warning
print("network graph saved")


## 6 · HTML Dashboard Export

In [ ]:
import base64

def img_b64(path):
    p = pathlib.Path(path)
    if not p.exists(): return ''
    return 'data:image/png;base64,' + base64.b64encode(p.read_bytes()).decode()

overreach_chart = img_b64('overreach_heatmap.png')
network_chart   = img_b64('entity_network.png')

def df_to_html_table(df, empty_msg='No entries yet.'):
    if df is None or df.empty: return f'<p class="empty-msg">{empty_msg}</p>'
    cols = df.columns.tolist()
    head = ''.join(f'<th>{c}</th>' for c in cols)
    rows = ''
    for _, row in df.iterrows():
        cells = ''
        for c in cols:
            val = str(row.get(c, '') or '')
            if val.startswith('http'):
                val = f'<a href="{val}" target="_blank">{val[:55]}...</a>' if len(val) > 55 else f'<a href="{val}" target="_blank">{val}</a>'
            cells += f'<td>{val}</td>'
        rows += f'<tr>{cells}</tr>'
    return f'<table><thead><tr>{head}</tr></thead><tbody>{rows}</tbody></table>'

def news_table_html(df, n=40):
    if df is None or df.empty: return '<p class="empty-msg">No news fetched this run.</p>'
    rows = ''
    for _, r in df.head(n).iterrows():
        ds = r['published'].strftime('%Y-%m-%d') if r.get('published') and hasattr(r['published'], 'strftime') else str(r.get('raw_date', ''))[:10]
        title = str(r.get('title', ''))
        link  = str(r.get('link', ''))
        title_a = f'<a href="{link}" target="_blank">{title}</a>' if link else title
        rows += f'<tr><td>{ds}</td><td><span class="cat-badge">{r.get("category", "")}</span></td><td>{r.get("source", "")}</td><td>{title_a}</td></tr>'
    return f'<table><thead><tr><th>Date</th><th>Category</th><th>Source</th><th>Headline</th></tr></thead><tbody>{rows}</tbody></table>'

# Build HTML using string concatenation — avoids backslash-in-f-string SyntaxError
A = '"'

stat_row = (
    '<div class=' + A + 'stat-row' + A + '>'
    + '<div class=' + A + 'stat-card' + A + '><div class=' + A + 'num' + A + '>' + str(len(court_df))    + '</div><div class=' + A + 'lbl' + A + '>Court Rulings</div></div>'
    + '<div class=' + A + 'stat-card' + A + '><div class=' + A + 'num' + A + '>' + str(len(incident_df)) + '</div><div class=' + A + 'lbl' + A + '>Incidents</div></div>'
    + '<div class=' + A + 'stat-card' + A + '><div class=' + A + 'num' + A + '>' + str(len(policy_df))   + '</div><div class=' + A + 'lbl' + A + '>Policy Actions</div></div>'
    + '<div class=' + A + 'stat-card' + A + '><div class=' + A + 'num' + A + '>' + str(len(news_df))     + '</div><div class=' + A + 'lbl' + A + '>News Items</div></div>'
    + '</div>'
)

chart_row = (
    '<div class=' + A + 'chart-row' + A + '>'
    + '<div class=' + A + 'chart-box' + A + '><h3>Overreach Category Frequency</h3><img src=' + A + overreach_chart + A + ' alt=' + A + 'Overreach Heatmap' + A + '/></div>'
    + '<div class=' + A + 'chart-box' + A + '><h3>Enforcement Entity Network</h3><img src=' + A + network_chart + A + ' alt=' + A + 'Entity Network' + A + '/></div>'
    + '</div>'
)

entity_cols = ['Abbreviation', 'Parent', 'Primary Role', 'Overreach Flags', 'Key Legal Constraints', 'Active Litigation']

sections = (
    '<section id=' + A + 'summary' + A + '><h2>Summary Stats</h2>' + stat_row + chart_row + '</section>'
    + '<section id=' + A + 'entities' + A + '><h2>Enforcement Entities</h2>' + df_to_html_table(entity_df[entity_cols]) + '</section>'
    + '<section id=' + A + 'courts' + A + '><h2>Court Rulings</h2>' + df_to_html_table(court_df) + '</section>'
    + '<section id=' + A + 'incidents' + A + '><h2>Enforcement Incidents</h2>' + df_to_html_table(incident_df) + '</section>'
    + '<section id=' + A + 'policy' + A + '><h2>Policy Actions</h2>' + df_to_html_table(policy_df) + '</section>'
    + '<section id=' + A + 'news' + A + '><h2>News Feed (' + str(len(news_df)) + ' items)</h2>' + news_table_html(news_df, 40) + '</section>'
)

nav = ('<nav>'
    + '<a href=' + A + '#summary' + A + '>Summary</a>'
    + '<a href=' + A + '#entities' + A + '>Entities</a>'
    + '<a href=' + A + '#courts' + A + '>Court Rulings</a>'
    + '<a href=' + A + '#incidents' + A + '>Incidents</a>'
    + '<a href=' + A + '#policy' + A + '>Policy Actions</a>'
    + '<a href=' + A + '#news' + A + '>News Feed</a>'
    + '</nav>')

css = '''
:root{--ink:#0e0d0b;--paper:#f5f0e8;--accent:#1a3a5c;--danger:#c0392b;--muted:#7a7468;--border:#d4cfc3;--surface:#fffef9}
*,*::before,*::after{box-sizing:border-box;margin:0;padding:0}
body{font-family:Georgia,serif;background:var(--paper);color:var(--ink);font-size:15px}
header{background:var(--accent);color:#fff;padding:1.5rem 2rem;display:flex;align-items:center;justify-content:space-between}
header h1{font-size:1.4rem}.meta{font-family:monospace;font-size:.72rem;opacity:.7}
.danger-bar{background:var(--danger);color:#fff;padding:.45rem 2rem;font-family:monospace;font-size:.72rem;letter-spacing:.06em}
nav{background:var(--ink);display:flex;gap:0;overflow-x:auto}
nav a{color:#ccc;font-family:monospace;font-size:.75rem;padding:.6rem 1.2rem;text-decoration:none;white-space:nowrap;border-right:1px solid #333}
nav a:hover{background:var(--accent);color:#fff}
.shell{max-width:1200px;margin:0 auto;padding:2rem}
section{margin-bottom:3rem}
section h2{font-size:1.05rem;font-weight:700;letter-spacing:.06em;text-transform:uppercase;color:var(--accent);border-bottom:2px solid var(--accent);padding-bottom:.4rem;margin-bottom:1.2rem}
table{width:100%;border-collapse:collapse;font-size:.8rem}
th{background:var(--accent);color:#fff;padding:.5rem .7rem;text-align:left;font-family:monospace;font-size:.72rem}
td{padding:.45rem .7rem;border-bottom:1px solid var(--border);vertical-align:top}
tr:nth-child(even) td{background:#f0ece2}
a{color:var(--accent)}
.cat-badge{background:var(--accent);color:#fff;font-size:.65rem;padding:.15em .5em;border-radius:2px;font-family:monospace}
.empty-msg{color:var(--muted);font-style:italic;padding:.5rem 0}
.chart-row{display:grid;grid-template-columns:1fr 1fr;gap:1.5rem;margin-bottom:2rem}
.chart-box{background:var(--surface);border:1px solid var(--border);border-radius:4px;padding:1rem}
.chart-box img{width:100%;height:auto}
.chart-box h3{font-family:monospace;font-size:.75rem;text-transform:uppercase;color:var(--muted);margin-bottom:.6rem}
.stat-row{display:grid;grid-template-columns:repeat(4,1fr);gap:1rem;margin-bottom:2rem}
.stat-card{background:var(--surface);border:1px solid var(--border);border-radius:4px;padding:1rem 1.2rem}
.stat-card .num{font-size:2rem;font-weight:700;color:var(--danger);font-family:monospace}
.stat-card .lbl{font-family:monospace;font-size:.68rem;color:var(--muted);text-transform:uppercase;letter-spacing:.06em}
footer{background:var(--ink);color:#888;font-family:monospace;font-size:.7rem;text-align:center;padding:1.2rem;margin-top:3rem}
@media(max-width:700px){.chart-row,.stat-row{grid-template-columns:1fr}}
'''

html = (
    '<!DOCTYPE html>\n'
    '<html lang=' + A + 'en' + A + '><head>'
    '<meta charset=' + A + 'utf-8' + A + '/>'
    '<meta name=' + A + 'viewport' + A + ' content=' + A + 'width=device-width,initial-scale=1' + A + '/>'
    '<title>ICE &amp; Immigration Law Monitor &mdash; ' + GENERATED_AT + '</title>'
    '<style>' + css + '</style></head>\n'
    '<body>\n'
    '<header><h1>ICE &amp; Immigration Law Monitor</h1>'
    '<span class=' + A + 'meta' + A + '>Generated: ' + GENERATED_AT + ' | NarroVue</span></header>\n'
    '<div class=' + A + 'danger-bar' + A + '>&#9888; OVERREACH MONITOR &mdash; tracking where enforcement pushes into illegality</div>\n'
    + nav + '\n'
    '<div class=' + A + 'shell' + A + '>\n'
    + sections + '\n'
    '</div>\n'
    '<footer>NarroVue &middot; ICE &amp; Immigration Law Monitor &middot; Generated ' + GENERATED_AT + '</footer>\n'
    '</body></html>'
)

OUTPUT_HTML.write_text(html, encoding='utf-8')
print(f'Dashboard exported -> {OUTPUT_HTML.resolve()}')
print(f'  File size: {OUTPUT_HTML.stat().st_size / 1024:.1f} KB')


## 7 · Manual Search Targets

In [ ]:
SEARCH_QUERIES = [
    "ICE warrantless raid 2026",
    "Alien Enemies Act court ruling 2026",
    "Trump immigration executive order enjoined",
    "ICE detention death 2026",
    "sanctuary city funding lawsuit 2026",
    "DACA SCOTUS ruling 2026",
    "birthright citizenship court order",
    "expedited removal constitutional challenge",
    "EOIR immigration court backlog 2026",
    "ICE courthouse arrest due process",
    "non-refoulement violation CBP 2026",
    "family separation injunction update",
    "Mahmoud Khalil immigration case update",
    "CECOT deportation habeas corpus",
    "CBP turnback asylum seeker 2026",
    "immigration attorney access denied ICE",
]

print("Suggested search queries:\n")
for i, q in enumerate(SEARCH_QUERIES, 1):
    print(f"  {i:2d}. {q}")

## 8 · Weekly Brief Export

In [ ]:
lines = []
lines.append("NarroVue · ICE & Immigration Law Monitor")
lines.append(f"Run date    : {GENERATED_AT}")
lines.append(f"Entities    : {len(entity_df)}")
lines.append(f"News items  : {len(news_df)}")
lines.append("")

lines.append(f"COURT RULINGS ({len(court_df)} entries)")
lines.append("-" * 56)
if court_df.empty:
    lines.append("  (none)")
else:
    for _, r in court_df.iterrows():
        lines.append(f"  {r.get('Date','')} | {r.get('Court','')} | {r.get('Case','')}")
        lines.append(f"    Subject: {r.get('Subject','')}")
        for ln in textwrap.wrap(r.get('Ruling',''), 72): lines.append(f"    → {ln}")
        lines.append(f"    Overreach: {r.get('Overreach Tags','')}")
        lines.append(f"    Outcome: {r.get('Outcome','')}")
        if r.get('Source'): lines.append(f"    Source: {r['Source']}")
        lines.append("")

lines.append(f"ENFORCEMENT INCIDENTS ({len(incident_df)} entries)")
lines.append("-" * 56)
if incident_df.empty:
    lines.append("  (none)")
else:
    for _, r in incident_df.iterrows():
        lines.append(f"  {r.get('Date','')} | {r.get('Entity','')} | {r.get('Location','')} | {r.get('Incident Type','')}")
        for ln in textwrap.wrap(r.get('Description',''), 72): lines.append(f"    → {ln}")
        lines.append(f"    Overreach: {r.get('Overreach Tags','')}")
        if r.get('Legal Challenge'): lines.append(f"    Legal: {r['Legal Challenge']}")
        lines.append("")

lines.append(f"POLICY ACTIONS ({len(policy_df)} entries)")
lines.append("-" * 56)
if policy_df.empty:
    lines.append("  (none)")
else:
    for _, r in policy_df.iterrows():
        lines.append(f"  {r.get('Date','')} | {r.get('Entity','')} | {r.get('Policy Area','')}")
        for ln in textwrap.wrap(r.get('Action',''), 72): lines.append(f"    Action: {ln}")
        lines.append(f"    Overreach: {r.get('Overreach Tags','')}")
        if r.get('Federal Interaction'):
            for ln in textwrap.wrap(r['Federal Interaction'], 72): lines.append(f"    Federal: {ln}")
        if r.get('States Affected'): lines.append(f"    States: {r['States Affected']}")
        lines.append("")

lines.append("TOP NEWS HEADLINES (up to 10 most recent)")
lines.append("-" * 56)
if news_df.empty:
    lines.append("  (no news fetched this run)")
else:
    for _, r in news_df.head(10).iterrows():
        ds = r['published'].strftime('%Y-%m-%d') if r.get('published') and hasattr(r['published'],'strftime') else str(r.get('raw_date',''))[:10]
        lines.append(f"  {ds} | {r.get('source','')} | {r.get('title','')}")
        lines.append(f"    {r.get('link','')}")
        lines.append("")

output_text = "\n".join(lines)
BRIEF_TXT.write_text(output_text, encoding="utf-8")
print(f"\n✓ Weekly brief input written → {BRIEF_TXT.resolve()}")
print(f"  File size: {BRIEF_TXT.stat().st_size} bytes")
print()
print(output_text[:800] + ("\n  …" if len(output_text) > 800 else ""))